# LearningMirror — Handwriting Model Training (Dysgraphia/Dyslexia indicators)

This notebook trains a **real convolutional neural network (CNN)** that looks at images of handwritten letters and classifies each one as:

- **Normal** — written the usual way
- **Reversal** — mirror-written / flipped (a classic dyslexia/dysgraphia indicator)
- **Corrected** — initially reversed, then fixed

**Dataset:** the *Synthetic Dyslexia Handwriting Dataset* (Kaggle: `michaelfink0923/synthetic-dyslexia-handwriting-dataset`, Apache 2.0). It contains 78,275 Normal, 52,196 Reversal and 8,029 Corrected letters, built from NIST Special Database 19, the A–Z Handwritten Alphabets set, and real handwriting from dyslexic primary-school children in Penang, Malaysia. It is in **YOLO format** (big images with a labelled box around every letter), so Step 3 crops each letter out before training.

**What you need:** just a Google account. The dataset is public — no Kaggle token needed. **Colab Pro** recommended, and set **Runtime → Change runtime type → GPU** (T4 is fine).

**How to run:** Runtime → **Run all**. Total time: roughly 30–50 minutes.

At the end you get a file **`learningmirror_handwriting_model.zip`** — send that back to Claude to wire into the website.

In [ ]:
#@title 1. Install tools
!pip install -q kagglehub tensorflowjs pyyaml
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
#@title 2. Download the dataset (public — no login needed)
import kagglehub
path = kagglehub.dataset_download("michaelfink0923/synthetic-dyslexia-handwriting-dataset")
print("Path to dataset files:", path)

# Show what's inside so we can see the YOLO structure
import os
for root, dirs, fs in os.walk(path):
    depth = root.replace(path, '').count(os.sep)
    if depth <= 2:
        print('  '*depth + os.path.basename(root) + '/', f'({len(fs)} files)' if fs else '')

In [ ]:
#@title 3. Read the YOLO labels and find class names
import os, glob, yaml

# 3a. class names: YOLO datasets usually ship a data.yaml / classes.txt
CLASS_NAMES_YOLO = None
for f in glob.glob(os.path.join(path, '**', '*.yaml'), recursive=True) + \
         glob.glob(os.path.join(path, '**', '*.yml'), recursive=True):
    try:
        y = yaml.safe_load(open(f))
        if isinstance(y, dict) and 'names' in y:
            n = y['names']
            CLASS_NAMES_YOLO = list(n.values()) if isinstance(n, dict) else list(n)
            print('Found classes in', os.path.basename(f), '->', CLASS_NAMES_YOLO)
            break
    except Exception:
        pass
if CLASS_NAMES_YOLO is None:
    for f in glob.glob(os.path.join(path, '**', 'classes.txt'), recursive=True):
        CLASS_NAMES_YOLO = [l.strip() for l in open(f) if l.strip()]
        print('Found classes in classes.txt ->', CLASS_NAMES_YOLO)
        break
if CLASS_NAMES_YOLO is None:
    CLASS_NAMES_YOLO = ['Normal', 'Reversal', 'Corrected']   # dataset's documented order fallback
    print('No class file found — assuming', CLASS_NAMES_YOLO)

# 3b. pair image files with their label .txt files
label_files = glob.glob(os.path.join(path, '**', 'labels', '**', '*.txt'), recursive=True) \
           or glob.glob(os.path.join(path, '**', '*.txt'), recursive=True)
def find_image_for(lbl):
    stem = os.path.splitext(os.path.basename(lbl))[0]
    cand_dirs = [os.path.dirname(lbl).replace('/labels', '/images'), os.path.dirname(lbl)]
    for d in cand_dirs:
        for ext in ('.png', '.jpg', '.jpeg'):
            p = os.path.join(d, stem + ext)
            if os.path.exists(p): return p
    return None
pairs = [(find_image_for(l), l) for l in label_files]
pairs = [p for p in pairs if p[0]]
print(f"Found {len(pairs)} labelled images")
assert pairs, 'No image/label pairs found — check the folder printout above and adjust paths'


In [ ]:
#@title 4. Crop every labelled letter into class folders (~ a few minutes)
from PIL import Image
import random
random.seed(42)

BASE = '/content/prepared'
!rm -rf {BASE}
CAP_PER_CLASS = 20000   # keeps training fast; raise if you want (Corrected has only ~8k anyway)
counts = {c: 0 for c in CLASS_NAMES_YOLO}
os.makedirs(BASE, exist_ok=True)
for c in CLASS_NAMES_YOLO: os.makedirs(os.path.join(BASE, c), exist_ok=True)

random.shuffle(pairs)
for img_path, lbl_path in pairs:
    if all(counts[c] >= CAP_PER_CLASS for c in CLASS_NAMES_YOLO): break
    try:
        im = Image.open(img_path).convert('L')
    except Exception:
        continue
    W, H = im.size
    for line in open(lbl_path):
        parts = line.split()
        if len(parts) < 5: continue
        cls = int(float(parts[0]))
        if cls >= len(CLASS_NAMES_YOLO): continue
        cname = CLASS_NAMES_YOLO[cls]
        if counts[cname] >= CAP_PER_CLASS: continue
        # YOLO format: class x_center y_center width height (all 0-1)
        xc, yc, w, h = (float(v) for v in parts[1:5])
        x1, y1 = int((xc - w/2) * W), int((yc - h/2) * H)
        x2, y2 = int((xc + w/2) * W), int((yc + h/2) * H)
        x1, y1 = max(0, x1), max(0, y1); x2, y2 = min(W, x2), min(H, y2)
        if x2 - x1 < 5 or y2 - y1 < 5: continue
        crop = im.crop((x1, y1, x2, y2))
        crop.save(os.path.join(BASE, cname, f"{cname}_{counts[cname]}.png"))
        counts[cname] += 1
print("Letter crops prepared:", counts)

In [ ]:
#@title 6. Load images as training/validation datasets
IMG_SIZE = 96
BATCH = 64

train_ds = tf.keras.utils.image_dataset_from_directory(
    BASE, validation_split=0.15, subset='training', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, color_mode='grayscale')
val_ds = tf.keras.utils.image_dataset_from_directory(
    BASE, validation_split=0.15, subset='validation', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, color_mode='grayscale')

CLASS_NAMES = train_ds.class_names
print("Classes:", CLASS_NAMES)

# Class weights so a bigger class doesn't dominate training
import numpy as np
n = np.array([counts[c] for c in CLASS_NAMES], dtype='float32')
class_weight = {i: float(n.sum()/(len(n)*n[i])) for i in range(len(n))}
print("Class weights:", class_weight)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 7. The model

A compact CNN trained **from scratch** (letters are simple shapes, so this beats a huge pretrained net here, and it stays small enough to run inside a web browser).

**Important detail:** the data augmentation uses small rotations and shifts but **NO horizontal flips** — flipping a normal letter would literally create a reversed letter and poison the labels.

In [ ]:
from tensorflow.keras import layers, models

augment = models.Sequential([
    layers.RandomRotation(0.03),          # ±~10 degrees
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomZoom(0.1),
])  # NO RandomFlip — flips would corrupt Normal/Reversed labels!

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Rescaling(1./255),
    augment,
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(CLASS_NAMES), activation='softmax'),
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
#@title 8. Train (the real part — watch val_accuracy climb)
EPOCHS = 12
history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
    ])

In [ ]:
#@title 9. Results — accuracy curves and confusion matrix
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history['accuracy'], label='train')
ax[0].plot(history.history['val_accuracy'], label='validation')
ax[0].set_title('Accuracy'); ax[0].legend()
ax[1].plot(history.history['loss'], label='train')
ax[1].plot(history.history['val_loss'], label='validation')
ax[1].set_title('Loss'); ax[1].legend()
plt.show()

# Confusion matrix on the validation set
y_true, y_pred = [], []
for x, y in val_ds:
    p = model.predict(x, verbose=0)
    y_true.extend(y.numpy()); y_pred.extend(p.argmax(1))
from sklearn.metrics import confusion_matrix, classification_report
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45)
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion matrix')
plt.tight_layout(); plt.show()

In [ ]:
#@title 10. Sanity check — look at 12 predictions with your own eyes
import random as rnd
plt.figure(figsize=(12, 7))
for x, y in val_ds.take(1):
    idxs = rnd.sample(range(len(y)), min(12, len(y)))
    preds = model.predict(x, verbose=0)
    for k, i in enumerate(idxs):
        plt.subplot(3, 4, k+1)
        plt.imshow(x[i].numpy().squeeze(), cmap='gray')
        ok = preds[i].argmax() == y[i].numpy()
        plt.title(f"true: {CLASS_NAMES[y[i]]}\npred: {CLASS_NAMES[preds[i].argmax()]} ({preds[i].max():.0%})",
                  color=('green' if ok else 'red'), fontsize=9)
        plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
#@title 11. Export — save the model + browser version + zip
import json, os

# Save the model in two formats (h5 is the most convertible)
model.save('/content/handwriting_model.h5')
model.save('/content/handwriting_model.keras')

# Try to convert for running inside a web browser.
# If this errors, DON'T WORRY — the zip still contains the model files
# and the conversion can be done later on another machine.
try:
    import tensorflowjs as tfjs
    tfjs.converters.save_keras_model(model, '/content/tfjs_model')
    print('Browser (TensorFlow.js) version created')
except Exception as e:
    print('Browser conversion skipped — will be done later. Reason:', e)

os.makedirs('/content/tfjs_model', exist_ok=True)
with open('/content/tfjs_model/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)

!cd /content && zip -r learningmirror_handwriting_model.zip tfjs_model handwriting_model.h5 handwriting_model.keras
from google.colab import files
files.download('/content/learningmirror_handwriting_model.zip')
print('DONE — send this zip to Claude to wire into the website.')

## 12. What you just did (for your research paper)

- Trained a CNN classifier on letter crops extracted from the **Synthetic Dyslexia Handwriting Dataset** (Kaggle, Apache 2.0): 78,275 Normal / 52,196 Reversal / 8,029 Corrected letters, built from NIST Special Database 19, the A–Z Handwritten Alphabets set, and real handwriting samples from dyslexic primary-school children in Penang, Malaysia.
- Converted the dataset's YOLO object-detection labels into per-letter classification crops (Step 4) — describe this preprocessing in your Methodology.
- Used class weighting for the heavy imbalance (Corrected is ~16x smaller than Normal) and augmentation **without horizontal flips** (a flip would turn a Normal letter into a Reversal and corrupt the labels — mention this, it shows you understood the data).
- Evaluated with a held-out validation set, per-class precision/recall, and a confusion matrix.
- Exported the model to TensorFlow.js so it runs **inside the parent's browser** — no server, works on low-end phones, and no child handwriting ever leaves the device (matches the privacy section of the mentor's project note).

**For your literature review:** this dataset has two companion papers — *"Explainable AI in Handwriting Detection for Dyslexia Using Transfer Learning"* (arXiv:2410.19821) and *"Explainable YOLO-Based Dyslexia Detection in Synthetic Handwriting Data"* (arXiv:2501.15263). Both are ideal Literature Review Log entries: same data, same goal, different methods — and their limitations sections give you your "gap".

**Known limitations to write down:** letter-level classification; largely synthetic composition; Latin script only (Devanagari = future work); source children are Malaysian, not Indian; a writer-level split would be a stricter test — discuss with your mentor in Boston.

**Next step:** send `learningmirror_handwriting_model.zip` back to Claude. The website's handwriting task will then segment the parent's photo into letters and run each one through YOUR model.